<a href="https://colab.research.google.com/github/DongAnYu/ai-tutor-rag-system/blob/main/notebooks/Advanced_Retriever.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Install Packages and Setup Variables


In [2]:
!pip install -q llama-index==0.14.0 llama-index-vector-stores-chroma==0.5.3 llama-index-llms-google-genai==0.5.0 \
                chromadb==1.0.21 llama-index-llms-openai==0.5.6 jedi==0.19.2

In [27]:
import os

# Set the following API Keys in the Python environment. Will be used later.
# os.environ["OPENAI_API_KEY"] = "[OPENAI_API_KEY]"
# os.environ["GOOGLE_API_KEY"] = "<YOUR_API_KEY>"

from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["GOOGLE_API_KEY"] = userdata.get('Google_api_key')

In [4]:
# Allows running asyncio in environments with an existing event loop, like Jupyter notebooks.
import nest_asyncio

nest_asyncio.apply()

## Language Model and Embedding Model

In [28]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import Settings
from llama_index.llms.openai import OpenAI
from llama_index.llms.google_genai import GoogleGenAI

# Settings.llm = OpenAI(model="gpt-5-mini")

Settings.llm = GoogleGenAI(model="gemini-2.5-flash", temperature=1)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

**Note: You can create a vector store from scratch using the code below, or you can load it from Hugging Face using the code provided in this notebook.**

# Create a Vector Store


In [6]:
import chromadb

# create client and a new collection
# chromadb.EphemeralClient saves data in-memory.
chroma_client = chromadb.PersistentClient(path="./ai_tutor_knowledge")
chroma_collection = chroma_client.create_collection("ai_tutor_knowledge")

In [7]:
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core.storage.storage_context import StorageContext

# Define a storage context object using the created vector database.
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# Load the Dataset (JSON)


## Download


In [8]:
from huggingface_hub import hf_hub_download
file_path = hf_hub_download(repo_id="jaiganesan/ai_tutor_knowledge", filename="ai_tutor_knowledge.jsonl",repo_type="dataset",local_dir="/content")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


ai_tutor_knowledge.jsonl: 0.00B [00:00, ?B/s]

# Read File


In [9]:
import json
with open(file_path, "r") as file:
    ai_tutor_knowledge = [json.loads(line) for line in file]

len(ai_tutor_knowledge)

762

# Convert to Document obj


In [10]:
from typing import List
from llama_index.core import Document

def create_docs_from_list(data_list: List[dict]) -> List[Document]:
    documents = []
    for data in data_list:
        documents.append(
            Document(
                doc_id=data["doc_id"],
                text=data["content"],
                metadata={  # type: ignore
                    "url": data["url"],
                    "title": data["name"],
                    "tokens": data["tokens"],
                    "source": data["source"],
                },
                excluded_llm_metadata_keys=[
                    "title",
                    "tokens",
                    "source",
                ],
                excluded_embed_metadata_keys=[
                    "url",
                    "tokens",
                    "source",
                ],
            )
        )
    return documents

doc = create_docs_from_list(ai_tutor_knowledge)

# Transforming


In [11]:
from llama_index.core.node_parser import SentenceWindowNodeParser

# create the sentence window node parser
node_parser = SentenceWindowNodeParser.from_defaults(
    window_size=3,
    include_metadata=True,
    window_metadata_key="window",
    original_text_metadata_key="original_text",
)

In [12]:
documents = [i for i in doc if i.metadata['tokens']<8000]
nodes = node_parser.get_nodes_from_documents(documents)

In [13]:
len(nodes)

22083

In [14]:
nodes[0]

TextNode(id_='2adfc522-824b-4bfb-ad77-414eb31b3d21', embedding=None, metadata={'url': 'https://towardsai.net/p/machine-learning/bert-huggingface-model-deployment-using-kubernetes-github-repo-03-07-2024', 'title': 'BERT HuggingFace Model Deployment using Kubernetes [ Github Repo]  03/07/2024', 'tokens': 768, 'source': 'tai_blog', 'window': 'Github Repo : https://github.com/vaibhawkhemka/ML-Umbrella/tree/main/MLops/Model_Deployment/Bert_Kubernetes_deployment   Model development is useless if you dont deploy it to production  which comes with a lot of issues of scalability and portability.    I have deployed a basic BERT model from the huggingface transformer on Kubernetes with the help of docker  which will give a feel of how to deploy and manage pods on production.    Model Serving and Deployment:ML Pipeline:Workflow:   Model server (using FastAPI  uvicorn) for BERT uncased model    Containerize model and inference scripts to create a docker image    Kubernetes deployment for these mode

## Index creation

In [29]:
from llama_index.core import VectorStoreIndex

# Add the documents to the database and create Index / embeddings
index = VectorStoreIndex(nodes, storage_context=storage_context,show_progress =True)

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

...

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/1603 [00:00<?, ?it/s]

In [30]:
# Compress the vector store directory to a zip file to be able to download and use later.
!zip -r vectorstore-windowed.zip ai_tutor_knowledge

  adding: ai_tutor_knowledge/ (stored 0%)
  adding: ai_tutor_knowledge/d9c7e276-3e0f-4db3-a517-04b12af28db8/ (stored 0%)
  adding: ai_tutor_knowledge/d9c7e276-3e0f-4db3-a517-04b12af28db8/link_lists.bin (deflated 78%)
  adding: ai_tutor_knowledge/d9c7e276-3e0f-4db3-a517-04b12af28db8/length.bin (deflated 79%)
  adding: ai_tutor_knowledge/d9c7e276-3e0f-4db3-a517-04b12af28db8/data_level0.bin (deflated 42%)
  adding: ai_tutor_knowledge/d9c7e276-3e0f-4db3-a517-04b12af28db8/header.bin (deflated 56%)
  adding: ai_tutor_knowledge/d9c7e276-3e0f-4db3-a517-04b12af28db8/index_metadata.pickle (deflated 44%)
  adding: ai_tutor_knowledge/chroma.sqlite3 (deflated 86%)


# Load Indexes


**Note: If you didn’t create the vector store from scratch, please uncomment the three code blocks/cells below.**

In [ ]:
# from huggingface_hub import hf_hub_download
# vectorstore = hf_hub_download(repo_id="jaiganesan/ai_tutor_knowledge", filename="vectorstore-windowed.zip",repo_type="dataset",local_dir="/content")

In [ ]:
# !unzip vectorstore-windowed.zip

In [ ]:
# import chromadb
# from llama_index.vector_stores.chroma import ChromaVectorStore
# from llama_index.core import VectorStoreIndex

# # Create your index
# db = chromadb.PersistentClient(path="./ai_tutor_knowledge")
# chroma_collection = db.get_or_create_collection("ai_tutor_knowledge")
# vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

# # Create your index
# index = VectorStoreIndex.from_vector_store(vector_store)

In [31]:
from llama_index.core.postprocessor import MetadataReplacementPostProcessor

query_engine = index.as_query_engine(
    llm=Settings.llm,
    # the target key defaults to `window` to match the node_parser's default
    node_postprocessors=[
        MetadataReplacementPostProcessor(target_metadata_key="window")
    ],
)

In [32]:
response = query_engine.query("Write about naive RAG and Speculative RAG?")
print(response)

Speculative RAG is a framework designed to enhance retrieval-augmented generation by utilizing a larger, generalist language model to efficiently verify multiple drafts produced in parallel by a smaller, distilled specialist language model. Each draft is created from a unique subset of retrieved documents, which provides varied perspectives on the evidence and helps to reduce the number of input tokens per draft. This method improves comprehension of each document subset and helps to lessen potential position bias that can occur with long contexts. The approach accelerates RAG by assigning the drafting task to the smaller specialist model, with the larger generalist model performing a single verification pass over the compiled drafts.

This method has demonstrated superior performance compared to standard RAG approaches, consistently outperforming them across various benchmarks, including TriviaQA, MusiQue, PubHealth, and ARC-Challenge. For example, a configuration of Speculative RAG u

In [33]:
for idx, item in enumerate(response.source_nodes):
    print("Source ", idx + 1)
    print("Original Text:", item.node.metadata["original_text"])
    print("Window:", item.node.metadata["window"])
    print("----")

Source  1
Original Text: In this work, we introduce SPECULATIVE RAG – a framework that leverages a larger generalist LM to efficiently verify multiple RAG drafts produced in parallel by a smaller, distilled specialist LM. 
Window: **Abstract**Retrieval augmented generation (RAG) combines the generative abilities of large language models (LLMs) with external knowledge sources to provide more accurate and up-to-date responses.  Recent RAG advancements focus on improving retrieval outcomes through iterative LLM refinement or self-critique capabilities acquired through additional instruction tuning of LLMs.  In this work, we introduce SPECULATIVE RAG – a framework that leverages a larger generalist LM to efficiently verify multiple RAG drafts produced in parallel by a smaller, distilled specialist LM.  Each draft is generated from a distinct subset of retrieved documents, offering diverse perspectives on the evidence while reducing input token counts per draft.  This approach enhances comp